# Section 1: Frequency-Domain Analysis

Visualize the spectral fingerprints of the three image sources we just
prepared in `00_data_pipeline.ipynb`.

**Research question 1**: do FFHQ (real), FaceForensics++ FaceSwap (GAN-style),
and Stable Diffusion v1.5 leave distinguishable artifacts in the DFT?

If they do, frequency-based features may help cross-generator transfer.
If they look identical at the population level, that signal is dead and
we lean entirely on learned spatial features.

## 1.0 Setup

In [ ]:
import os
import sys
from pathlib import Path

# When running from notebooks/ directory, hop up to project root.
PROJECT_ROOT = Path(".").resolve().parent
if (PROJECT_ROOT / "src").exists():
    sys.path.insert(0, str(PROJECT_ROOT))
    os.chdir(PROJECT_ROOT)
print(f"Working dir: {os.getcwd()}")

import numpy as np
import matplotlib.pyplot as plt
import cv2

from src.analysis.frequency_visualization import (
    compute_magnitude_spectrum,
    compute_azimuthal_average,
    plot_spectrum_comparison,
    plot_radial_profiles,
    batch_spectrum_analysis,
)

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-paper")
except OSError:
    plt.style.use("default")
plt.rcParams["figure.dpi"] = 120

In [ ]:
# Three sources produced by the data pipeline notebook.
SOURCES = [
    ("data/real",            "FFHQ (real)"),
    ("data/faceswap",        "FaceSwap (FF++ c23)"),
    ("data/diffusion_sd15",  "Stable Diffusion v1.5"),
]
FIGURES_DIR = Path("results/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

for src_dir, label in SOURCES:
    n = len(list(Path(src_dir).glob("*.png")))
    print(f"  {label:<25} {n} images at {src_dir}")
    assert n > 0, f"No images in {src_dir}. Run 00_data_pipeline first."

## 1.1 Single-image magnitude spectra

One representative image from each source, side-by-side. Per-image spectra
are noisy — useful for spotting strong artifacts but not for population
claims (see 1.3 for averaged versions).

In [ ]:
rng = np.random.default_rng(42)

samples = []
for src_dir, label in SOURCES:
    files = sorted(Path(src_dir).glob("*.png"))
    pick = files[rng.integers(0, len(files))]
    img = cv2.imread(str(pick), cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (256, 256))
    samples.append((img, label))

fig = plot_spectrum_comparison(
    samples,
    save_path=str(FIGURES_DIR / "spectra_single.png"),
    figsize=(15, 4),
)
plt.show()

## 1.2 Radial power profiles (single images)

The 2D spectrum collapsed to a 1D curve by averaging over angle. Differences
in mid-to-high frequency bands are the main thing to look for.

In [ ]:
fig = plot_radial_profiles(
    samples,
    save_path=str(FIGURES_DIR / "radial_single.png"),
    figsize=(8, 5),
)
plt.show()

## 1.3 Averaged spectra over many samples

Per-source averages (default 50 images each) suppress per-image noise and
expose systematic frequency artifacts. This is the more publishable view.

In [ ]:
N_SAMPLES = 200  # bump for more stable averages

avg_spectra = {}
for src_dir, label in SOURCES:
    print(f"Averaging {N_SAMPLES} spectra from {label}...")
    avg = batch_spectrum_analysis(
        src_dir,
        n_samples=N_SAMPLES,
        save_path=str(FIGURES_DIR / f"avg_spectrum_{Path(src_dir).name}.png"),
    )
    avg_spectra[label] = avg

In [ ]:
# Side-by-side comparison of averaged spectra (shared color scale)
fig, axes = plt.subplots(1, len(avg_spectra), figsize=(15, 4))
vmin = min(s.min() for s in avg_spectra.values())
vmax = max(s.max() for s in avg_spectra.values())

for ax, (label, spec) in zip(axes, avg_spectra.items()):
    im = ax.imshow(spec, cmap="magma", vmin=vmin, vmax=vmax)
    ax.set_title(label)
    ax.axis("off")
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
fig.suptitle(f"Averaged DFT Magnitude Spectra (n={N_SAMPLES} per source)")
plt.savefig(FIGURES_DIR / "avg_spectra_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 1.4 Averaged radial profiles

1D summary of 1.3. If the curves diverge in any frequency band, a frequency-
based classifier would have a real signal there.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for label, spec in avg_spectra.items():
    radial = compute_azimuthal_average(spec)
    freqs = np.arange(len(radial)) / len(radial)
    ax.plot(freqs, radial, label=label, linewidth=1.6)
ax.set_xlabel("Normalized frequency")
ax.set_ylabel("Mean log-magnitude")
ax.set_title(f"Averaged radial power profile (n={N_SAMPLES} per source)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "avg_radial_profiles.png", dpi=150, bbox_inches="tight")
plt.show()

## 1.5 What to look for

- **Real (FFHQ)**: smooth 1/f-style decay, no periodic peaks.
- **FaceSwap (GAN-based)**: expect checkerboard-like high-frequency peaks
  from transposed-conv upsampling. Should diverge from real most clearly
  in the upper third of the radial profile.
- **Stable Diffusion v1.5**: artifacts are subtler than GAN. Look for
  mid-frequency bumps; the curve often sits *between* real and GAN.

If FaceSwap and SD are visibly distinct in the radial plot, the cross-
generator transfer problem is hard precisely because their fingerprints
live in different frequency bands — a model trained to spot one won't
automatically catch the other. That's the experiment in notebook 03.